In [ ]:
# import warnings
# from scipy.sparse import SparseEfficiencyWarning

# warnings.filterwarnings(
#     "ignore",
#     category=SparseEfficiencyWarning
# )

In [2]:
import itertools
import time
import numpy as np
import pandas as pd

from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA

from qiskit.primitives import StatevectorSampler

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer

In [3]:
def validate_qubo_matrix(Q):
    Q = np.asarray(Q, dtype=float)

    if Q.ndim != 2:
        raise ValueError("Q must be a two-dimensional matrix.")

    if Q.shape[0] != Q.shape[1]:
        raise ValueError("Q must be square.")

    if not np.all(np.isfinite(Q)):
        raise ValueError("Q must contain only finite real numbers.")

    return Q

def qubo_energy(Q, x):
    Q = validate_qubo_matrix(Q)
    x = np.asarray(x, dtype=int)
    return float(x @ Q @ x)

def qubo_matrix_to_quadratic_program(Q, name="qubo_problem"):
    Q = validate_qubo_matrix(Q)
    n = Q.shape[0]

    qp = QuadraticProgram(name)

    for i in range(n):
        qp.binary_var(name=f"x_{i}")

    linear = {}
    quadratic = {}

    for i in range(n):
        if abs(Q[i, i]) > 1e-12:
            linear[f"x_{i}"] = float(Q[i, i])

    for i in range(n):
        for j in range(i + 1, n):
            coeff = float(Q[i, j] + Q[j, i])
            if abs(coeff) > 1e-12:
                quadratic[(f"x_{i}", f"x_{j}")] = coeff

    qp.minimize(linear=linear, quadratic=quadratic)

    return qp

def brute_force_qubo(Q):
    Q = validate_qubo_matrix(Q)
    n = Q.shape[0]

    best_x = None
    best_energy = np.inf
    all_results = []

    for bits in itertools.product([0, 1], repeat=n):
        x = np.array(bits, dtype=int)
        energy = qubo_energy(Q, x)

        all_results.append((x, energy))

        if energy < best_energy:
            best_energy = energy
            best_x = x.copy()

    return best_x, best_energy, all_results

def solve_relaxed_qubo(Q, n_starts=20, seed=123):
    Q = validate_qubo_matrix(Q)
    n = Q.shape[0]

    Qsym = 0.5 * (Q + Q.T)

    def objective(c):
        return float(c @ Q @ c)

    def gradient(c):
        return 2.0 * Qsym @ c

    bounds = [(0.0, 1.0) for _ in range(n)]

    rng = np.random.default_rng(seed)

    initial_points = [np.full(n, 0.5)]
    initial_points += [rng.random(n) for _ in range(max(0, n_starts - 1))]

    best_result = None

    for c0 in initial_points:
        result = minimize(
            objective,
            c0,
            jac=gradient,
            bounds=bounds,
            method="L-BFGS-B",
        )

        if best_result is None or result.fun < best_result.fun:
            best_result = result

    c_best = np.clip(best_result.x, 0.0, 1.0)
    relaxed_energy = float(c_best @ Q @ c_best)

    return c_best, relaxed_energy, best_result


def warm_start_angles(c, epsilon=1e-3):
    c = np.asarray(c, dtype=float)

    if c.ndim != 1:
        raise ValueError("c must be a one-dimensional vector.")

    if not np.all(np.isfinite(c)):
        raise ValueError("c must contain only finite values.")

    if not np.all((0.0 <= c) & (c <= 1.0)):
        raise ValueError("All c values must lie in [0, 1].")

    if not (0.0 <= epsilon < 0.5):
        raise ValueError("epsilon must satisfy 0 <= epsilon < 0.5.")

    c_clipped = np.clip(c, epsilon, 1.0 - epsilon)

    return 2.0 * np.arcsin(np.sqrt(c_clipped))


def build_warm_start_initial_state(c, epsilon=1e-3):
    theta = warm_start_angles(c, epsilon=epsilon)
    n = len(theta)

    initial_state = QuantumCircuit(n, name="warm_start_initial_state")

    for i, theta_i in enumerate(theta):
        initial_state.ry(theta_i, i)

    return initial_state


def build_warm_start_mixer(c, epsilon=1e-3):
    theta = warm_start_angles(c, epsilon=epsilon)
    n = len(theta)

    beta = Parameter("β")
    mixer = QuantumCircuit(n, name="warm_start_mixer")

    for i, theta_i in enumerate(theta):
        mixer.ry(-theta_i, i)
        mixer.rz(-2.0 * beta, i)
        mixer.ry(theta_i, i)

    return mixer


def solve_qubo_with_standard_qaoa(
    Q,
    reps=1,
    maxiter=200,
    seed=123,
    initial_point=None,
):
    Q = validate_qubo_matrix(Q)
    qp = qubo_matrix_to_quadratic_program(Q)

    sampler = StatevectorSampler(seed=seed)
    optimizer = COBYLA(maxiter=maxiter)

    qaoa = QAOA(
        sampler=sampler,
        optimizer=optimizer,
        reps=reps,
        initial_point=initial_point,
    )

    optimizer_wrapper = MinimumEigenOptimizer(qaoa)

    start = time.perf_counter()
    result = optimizer_wrapper.solve(qp)
    elapsed = time.perf_counter() - start

    x = np.array(result.x, dtype=int)
    energy = qubo_energy(Q, x)

    return {
        "method": "standard_qaoa",
        "solution": x,
        "qubo_energy": energy,
        "qiskit_result": result,
        "elapsed_seconds": elapsed,
        "quadratic_program": qp,
    }


def solve_qubo_with_warm_start_qaoa(
    Q,
    reps=1,
    maxiter=200,
    seed=123,
    n_starts=20,
    epsilon=1e-3,
    initial_point=None,
):
    Q = validate_qubo_matrix(Q)
    qp = qubo_matrix_to_quadratic_program(Q)

    c_star, relaxed_energy, relaxed_result = solve_relaxed_qubo(
        Q=Q,
        n_starts=n_starts,
        seed=seed,
    )

    initial_state = build_warm_start_initial_state(
        c=c_star,
        epsilon=epsilon,
    )

    mixer = build_warm_start_mixer(
        c=c_star,
        epsilon=epsilon,
    )

    sampler = StatevectorSampler(seed=seed)
    optimizer = COBYLA(maxiter=maxiter)

    qaoa = QAOA(
        sampler=sampler,
        optimizer=optimizer,
        reps=reps,
        initial_state=initial_state,
        mixer=mixer,
        initial_point=initial_point,
    )

    optimizer_wrapper = MinimumEigenOptimizer(qaoa)

    start = time.perf_counter()
    result = optimizer_wrapper.solve(qp)
    elapsed = time.perf_counter() - start

    x = np.array(result.x, dtype=int)
    energy = qubo_energy(Q, x)

    return {
        "method": "warm_start_qaoa",
        "solution": x,
        "qubo_energy": energy,
        "relaxed_solution": c_star,
        "relaxed_energy": relaxed_energy,
        "relaxed_result": relaxed_result,
        "initial_state": initial_state,
        "mixer": mixer,
        "qiskit_result": result,
        "elapsed_seconds": elapsed,
        "quadratic_program": qp,
    }


def bitstring_from_array(x):
    return "".join(str(int(v)) for v in x)


def probability_of_bitstring(result, target_x):
    if not hasattr(result, "samples") or result.samples is None:
        return np.nan

    target_x = np.array(target_x, dtype=int)
    prob = 0.0

    for sample in result.samples:
        sample_x = np.array(sample.x, dtype=int)
        if np.array_equal(sample_x, target_x):
            prob += float(getattr(sample, "probability", 0.0))

    return prob


def benchmark_qaoa_methods(
    Q,
    reps_list=(1, 2),
    seeds=(123, 456, 789),
    maxiter=200,
    n_starts=20,
    epsilon=1e-3,
):
    Q = validate_qubo_matrix(Q)

    exact_x, exact_energy, _ = brute_force_qubo(Q)

    rows = []

    for reps in reps_list:
        for seed in seeds:
            standard_out = solve_qubo_with_standard_qaoa(
                Q=Q,
                reps=reps,
                maxiter=maxiter,
                seed=seed,
            )

            warm_out = solve_qubo_with_warm_start_qaoa(
                Q=Q,
                reps=reps,
                maxiter=maxiter,
                seed=seed,
                n_starts=n_starts,
                epsilon=epsilon,
            )

            for out in [standard_out, warm_out]:
                x = out["solution"]
                energy = qubo_energy(Q, x)

                prob_exact = probability_of_bitstring(
                    out["qiskit_result"],
                    exact_x,
                )

                row = {
                    "method": out["method"],
                    "reps": reps,
                    "seed": seed,
                    "solution": bitstring_from_array(x),
                    "energy": energy,
                    "exact_energy": exact_energy,
                    "energy_gap": energy - exact_energy,
                    "found_exact": np.isclose(energy, exact_energy),
                    "probability_exact": prob_exact,
                    "elapsed_seconds": out["elapsed_seconds"],
                }

                if out["method"] == "warm_start_qaoa":
                    row["relaxed_energy"] = out["relaxed_energy"]
                    row["relaxed_solution"] = np.array2string(
                        out["relaxed_solution"],
                        precision=3,
                    )
                else:
                    row["relaxed_energy"] = np.nan
                    row["relaxed_solution"] = ""

                rows.append(row)

    return pd.DataFrame(rows)

In [4]:
Q = np.array([
        [1, -2,  0],
        [0,  1, -2],
        [0,  0,  1],
    ], dtype=float)

exact_x, exact_energy, _ = brute_force_qubo(Q)

standard_out = solve_qubo_with_standard_qaoa(
        Q=Q,
        reps=1,
        maxiter=200,
        seed=123,
    )

warm_out = solve_qubo_with_warm_start_qaoa(
        Q=Q,
        reps=1,
        maxiter=200,
        seed=123,
        n_starts=20,
        epsilon=1e-3,
    )

print("Exact solution:", exact_x)
print("Exact energy:", exact_energy)

print("Standard QAOA solution:", standard_out["solution"])
print("Standard QAOA energy:", standard_out["qubo_energy"])

print("Relaxed solution c*:", warm_out["relaxed_solution"])
print("Relaxed energy:", warm_out["relaxed_energy"])

print("Warm-start QAOA solution:", warm_out["solution"])
print("Warm-start QAOA energy:", warm_out["qubo_energy"])

Exact solution: [1 1 1]
Exact energy: -1.0
Standard QAOA solution: [1 1 1]
Standard QAOA energy: -1.0
Relaxed solution c*: [1. 1. 1.]
Relaxed energy: -1.0
Warm-start QAOA solution: [1 1 1]
Warm-start QAOA energy: -1.0


In [5]:
benchmark_df = benchmark_qaoa_methods(
        Q=Q,
        reps_list=(1, 2),
        seeds=(123, 456, 789),
        maxiter=200,
        n_starts=20,
        epsilon=1e-3,
    )

print("\nBenchmark:")
print(benchmark_df)

summary = benchmark_df.groupby(["method", "reps"]).agg(
        success_rate=("found_exact", "mean"),
        mean_energy=("energy", "mean"),
        best_energy=("energy", "min"),
        mean_gap=("energy_gap", "mean"),
        mean_probability_exact=("probability_exact", "mean"),
        mean_elapsed_seconds=("elapsed_seconds", "mean"),
    ).reset_index()

print("\nBenchmark summary:")
print(summary)


Benchmark:
             method  reps  seed solution  energy  exact_energy  energy_gap  \
0     standard_qaoa     1   123      111    -1.0          -1.0         0.0   
1   warm_start_qaoa     1   123      111    -1.0          -1.0         0.0   
2     standard_qaoa     1   456      111    -1.0          -1.0         0.0   
3   warm_start_qaoa     1   456      111    -1.0          -1.0         0.0   
4     standard_qaoa     1   789      111    -1.0          -1.0         0.0   
5   warm_start_qaoa     1   789      111    -1.0          -1.0         0.0   
6     standard_qaoa     2   123      111    -1.0          -1.0         0.0   
7   warm_start_qaoa     2   123      111    -1.0          -1.0         0.0   
8     standard_qaoa     2   456      111    -1.0          -1.0         0.0   
9   warm_start_qaoa     2   456      111    -1.0          -1.0         0.0   
10    standard_qaoa     2   789      111    -1.0          -1.0         0.0   
11  warm_start_qaoa     2   789      111    -1.0    

In [6]:
Q = np.array([
    [-3,  2, -1,  0,  2,  0, -2,  1],
    [ 0, -2,  3, -2,  0,  1,  0, -1],
    [ 0,  0, -4,  2, -2,  0,  1,  0],
    [ 0,  0,  0, -1,  3, -3,  0,  2],
    [ 0,  0,  0,  0, -3,  2, -1,  0],
    [ 0,  0,  0,  0,  0, -2,  2, -2],
    [ 0,  0,  0,  0,  0,  0, -3,  1],
    [ 0,  0,  0,  0,  0,  0,  0, -2],
], dtype=float)

exact_x, exact_energy, _ = brute_force_qubo(Q)

standard_out = solve_qubo_with_standard_qaoa(
        Q=Q,
        reps=3,
        maxiter=1000,
        seed=123,
    )

warm_out = solve_qubo_with_warm_start_qaoa(
        Q=Q,
        reps=3,
        maxiter=1000,
        seed=123,
        n_starts=20,
        epsilon=1e-3,
    )

print("Exact solution:", exact_x)
print("Exact energy:", exact_energy)

print("Standard QAOA solution:", standard_out["solution"])
print("Standard QAOA energy:", standard_out["qubo_energy"])

print("Relaxed solution c*:", warm_out["relaxed_solution"])
print("Relaxed energy:", warm_out["relaxed_energy"])

print("Warm-start QAOA solution:", warm_out["solution"])
print("Warm-start QAOA energy:", warm_out["qubo_energy"])

Exact solution: [1 0 1 0 1 0 1 0]
Exact energy: -16.0
Standard QAOA solution: [1 0 1 0 1 0 1 0]
Standard QAOA energy: -16.0
Relaxed solution c*: [1. 0. 1. 0. 1. 1. 1. 1.]
Relaxed energy: -16.0
Warm-start QAOA solution: [1 0 1 0 1 1 1 1]
Warm-start QAOA energy: -16.0
